In [ ]:
import pandas as pd
import os
import re

In [ ]:
print("=== 1단계: 데이터 불러오기 및 병합 ===")
# 게임별 장르 매핑
game_meta = {
    "Elden_Ring": "RPG", "Witcher_3": "RPG", "Cyberpunk_2077": "RPG",
    "PUBG": "FPS", "CS2": "FPS", "R6_Siege": "FPS"
}

df_list = []
for file in os.listdir("data"):
    if file.endswith("_reviews.csv"):
        game_name = file.replace("_reviews.csv", "")
        temp_df = pd.read_csv(f"data/{file}")
        temp_df['Game_Name'] = game_name
        temp_df['Genre'] = game_meta.get(game_name, "Unknown")
        df_list.append(temp_df)

In [ ]:
# 하나로 병합
df = pd.concat(df_list, ignore_index=True)
print(f"병합 직후 데이터 크기: {df.shape}")

In [ ]:
print("\n=== 2단계: 데이터 정제 (결측치, 중복값, 타입 처리) ===")
df.dropna(subset=['review'], inplace=True)
df.drop_duplicates(subset=['review'], keep='first', inplace=True)
df['voted_up'] = df['voted_up'].astype(int) # True/False를 1/0으로 변환
print(f"정제 후 남은 데이터 크기: {df.shape}")


In [ ]:
print("\n=== 3단계: 자연어 처리(NLP) 및 파생 변수 생성 ===")
def clean_text(text):
    text = str(text).lower() # 소문자 변환
    text = re.sub(r'[^a-z\s]', '', text) # 특수문자 제거
    text = re.sub(r'\s+', ' ', text).strip() # 다중 공백 압축
    return text

df['clean_review'] = df['review'].apply(clean_text)
df['review_length'] = df['clean_review'].apply(lambda x: len(x.split()))

# 짧은 리뷰(단어 3개 미만) 필터링
df = df[df['review_length'] >= 3]
print(f"최종 전처리 완료 데이터 크기: {df.shape}")

In [ ]:
print("\n=== 4단계: 전처리 완료 데이터 백업 저장 ===")
df.to_csv("data/preprocessed_reviews.csv", index=False, encoding="utf-8-sig")
print("성공적으로 'preprocessed_reviews.csv' 파일이 저장되었습니다!")

# 결과 미리보기
display(df[['Game_Name', 'Genre', 'voted_up', 'review_length', 'clean_review']].head())

In [ ]:
import matplotlib.pyplot as plt

# 장르별 긍정(1)/부정(0) 개수 계산
rpg_pos = len(df[(df['Genre'] == 'RPG') & (df['voted_up'] == 1)])
rpg_neg = len(df[(df['Genre'] == 'RPG') & (df['voted_up'] == 0)])
rpg_stats = [rpg_pos, rpg_neg]

fps_pos = len(df[(df['Genre'] == 'FPS') & (df['voted_up'] == 1)])
fps_neg = len(df[(df['Genre'] == 'FPS') & (df['voted_up'] == 0)])
fps_stats = [fps_pos, fps_neg]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# RPG 파이 차트
ax1.pie(rpg_stats, labels=['Positive', 'Negative'], autopct='%1.1f%%', 
        startangle=90, colors=['#66b3ff','#ff9999'], explode=(0.05, 0))
ax1.set_title(f'RPG Genre Sentiment Ratio\n(Total: {rpg_pos+rpg_neg})', fontsize=14, fontweight='bold')

# FPS 파이 차트
ax2.pie(fps_stats, labels=['Positive', 'Negative'], autopct='%1.1f%%', 
        startangle=90, colors=['#66b3ff','#ff9999'], explode=(0.05, 0))
ax2.set_title(f'FPS Genre Sentiment Ratio\n(Total: {fps_pos+fps_neg})', fontsize=14, fontweight='bold')

# 그래프 한 번에 출력하기
plt.tight_layout()
plt.show()

In [ ]:
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

# 불용어 커스텀 설정
# 'game', 'play' 같은 단어는 게임 리뷰에 무조건 등장하므로 이를 제외
custom_stopwords = set(STOPWORDS)
custom_stopwords.update(['game', 'play', 'played', 'playing', 'even', 'one', 'really', 'make', 'time', 'much', 'will', 'get'])

# 장르별로 텍스트 하나로 뭉치기
rpg_text = " ".join(df[df['Genre'] == 'RPG']['clean_review'].astype(str))
fps_text = " ".join(df[df['Genre'] == 'FPS']['clean_review'].astype(str))

# 워드클라우드 이미지 생성 (RPG는 파란색, FPS는 빨간색)
wordcloud_rpg = WordCloud(width=800, height=400, background_color='white', 
                          stopwords=custom_stopwords, colormap='Blues').generate(rpg_text)

wordcloud_fps = WordCloud(width=800, height=400, background_color='white', 
                          stopwords=custom_stopwords, colormap='Reds').generate(fps_text)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# RPG 영역 (왼쪽)
ax1.imshow(wordcloud_rpg, interpolation='bilinear')
ax1.axis('off') # 지저분한 테두리와 눈금선 숨기기
ax1.set_title('RPG Genre Keywords', fontsize=18, fontweight='bold')

# FPS 영역 (오른쪽)
ax2.imshow(wordcloud_fps, interpolation='bilinear')
ax2.axis('off')
ax2.set_title('FPS Genre Keywords', fontsize=18, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# PyTorch 딥러닝 환경 및 scikit-learn 패키지 설치
%pip install torch torchvision scikit-learn

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np

print("=== 1단계: 단어 사전(Vocabulary) 구축 ===")
# 모든 리뷰의 단어를 모아서 고유한 번호 부여
vocab = {"<PAD>": 0, "<UNK>": 1} # 0: 빈칸 채움, 1: 모르는 단어
word_idx = 2

for review in df['clean_review']:
    for word in str(review).split():
        if word not in vocab:
            vocab[word] = word_idx
            word_idx += 1

print(f"완성된 총 단어 개수(Vocab Size): {len(vocab)}개")

print("\n=== 2단계: 텍스트를 숫자로 변환하고 길이 맞추기 ===")
MAX_LEN = 50 # 하나의 리뷰당 최대 50단어까지만 사용 (넘으면 자르고, 모자라면 0으로 채움)

def text_to_tensor(text):
    tokens = str(text).split()
    # 단어를 번호로 바꾸되, 사전에 없으면 <UNK>(1)로 처리
    seq = [vocab.get(word, 1) for word in tokens]
    
    # 50단어로 길이 맞추기
    if len(seq) > MAX_LEN:
        seq = seq[:MAX_LEN]
    else:
        seq = seq + [0] * (MAX_LEN - len(seq))
    return seq

# 전체 데이터 변환
X = np.array([text_to_tensor(text) for text in df['clean_review']])
y = np.array(df['voted_up'].values)

print("\n=== 3단계: 학습용(Train) / 테스트용(Test) 데이터 분리 ===")
# 전체 데이터를 8:2 비율로 무작위 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n=== 4단계: PyTorch Dataset & DataLoader 생성 ===")
class SteamReviewDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X_data = torch.LongTensor(X_data)
        self.y_data = torch.FloatTensor(y_data) # 이진 분류(0 or 1)이므로 Float 사용
        
    def __len__(self):
        return len(self.X_data)
    
    def __getitem__(self, idx):
        return self.X_data[idx], self.y_data[idx]

train_dataset = SteamReviewDataset(X_train, y_train)
test_dataset = SteamReviewDataset(X_test, y_test)

# 미니 배치 사이즈 64로 설정하여 데이터로더 장전
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"학습 데이터 개수: {len(train_dataset)}")
print(f"테스트 데이터 개수: {len(test_dataset)}")

# 로더가 잘 작동하는지 첫 번째 배치 뽑아서 확인
sample_X, sample_y = next(iter(train_loader))
print(f"배치 X(입력) 형태: {sample_X.shape} -> (배치 크기, 최대 단어 수)")
print(f"배치 y(정답) 형태: {sample_y.shape} -> (배치 크기)")

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# 1D CNN 모델 클래스 정의
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, filter_sizes):
        super(TextCNN, self).__init__()
        # 단어 번호를 의미 있는 연속 벡터로 변환
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # 여러 크기(예: 3단어, 4단어, 5단어씩 묶어서 보기)의 필터 생성
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        
        # 긍정(1)/부정(0) 확률을 뱉어낼 마지막 출력층
        self.fc = nn.Linear(len(filter_sizes) * num_filters, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x) # [배치크기, 단어수, 임베딩차원]
        embedded = embedded.permute(0, 2, 1) # Conv1d를 위해 차원 순서 변경
        
        # 합성곱과 최대 풀링 적용
        conved = [F.relu(conv(embedded)) for conv in self.convs]
        pooled = [F.max_pool1d(conv, conv.shape[2]).squeeze(2) for conv in conved]
        
        cat = torch.cat(pooled, dim=1)
        output = self.fc(cat)
        return self.sigmoid(output).squeeze(1)

# 하이퍼파라미터 및 훈련 세팅
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 학습 장치: {device}")

INPUT_DIM = len(vocab)
EMBEDING_DIM = 128
NUM_FILTERS = 100
FILTER_SIZES = [3, 4, 5]

model = TextCNN(INPUT_DIM, EMBEDING_DIM, NUM_FILTERS, FILTER_SIZES).to(device)
criterion = nn.BCELoss() # 이진 분류용 손실 함수
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 훈련 루프 시작
EPOCHS = 5

print("\n=== 모델 학습 시작 ===")
for epoch in range(EPOCHS):
    epoch_loss = 0
    epoch_acc = 0
    model.train() # 모델을 훈련 모드로 설정
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad() # 기울기 초기화
        predictions = model(batch_X) # 예측 수행
        loss = criterion(predictions, batch_y) # 오차 계산
        
        loss.backward() # 역전파
        optimizer.step() # 가중치 업데이트
        
        # 정확도 계산
        rounded_preds = torch.round(predictions)
        correct = (rounded_preds == batch_y).float()
        acc = correct.sum() / len(correct)
        
        epoch_loss += loss.item()
        epoch_acc += acc.item()
        
    print(f"Epoch: {epoch+1:02} | Loss: {epoch_loss/len(train_loader):.4f} | Acc: {epoch_acc/len(train_loader)*100:.2f}%")

In [ ]:
import torch

print("=== 1단계: 테스트 데이터(Unseen Data) 최종 평가 ===")
# 모델을 평가 모드로 전환 (Dropout, Batchnorm 등 고정)
model.eval()
test_loss = 0
test_acc = 0

# 기울기 계산 비활성화
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        
        rounded_preds = torch.round(predictions)
        correct = (rounded_preds == batch_y).float()
        acc = correct.sum() / len(correct)
        
        test_loss += loss.item()
        test_acc += acc.item()

final_test_loss = test_loss / len(test_loader)
final_test_acc = test_acc / len(test_loader) * 100

print(f"👉 최종 테스트 정확도: {final_test_acc:.2f}% (Loss: {final_test_loss:.4f})")
print("--------------------------------------------------\n")


print("=== 2단계: 직접 쓴 리뷰로 인공지능 테스트해 보기 ===")
# 새로운 리뷰 텍스트가 들어오면 전처리 -> 텐서 변환 -> 모델 예측을 수행하는 함수
def predict_sentiment(review_text):
    model.eval()
    
    # 1. 텍스트 정제 (소문자, 특수문자 제거)
    cleaned_text = clean_text(review_text)
    
    # 2. 숫자로 변환하고 길이(50) 맞추기
    tensor_input = text_to_tensor(cleaned_text)
    
    # 3. 파이토치 텐서로 변환하고 배치 차원(1) 추가 -> [1, 50] 형태
    tensor_input = torch.LongTensor(tensor_input).unsqueeze(0).to(device)
    
    # 4. 모델 예측
    with torch.no_grad():
        prediction = model(tensor_input).item()
    
    # 5. 결과 출력
    sentiment = "🟢 긍정 (Positive)" if prediction >= 0.5 else "🔴 부정 (Negative)"
    print(f"입력하신 리뷰: '{review_text}'")
    print(f"분석 결과: {sentiment} (긍정일 확률: {prediction*100:.1f}%)")
    print("-" * 40)

# 가상의 유저 리뷰를 직접 넣어서 테스트
predict_sentiment("This game is an absolute masterpiece. The story is amazing!") # 극찬 (RPG 유저 느낌)
predict_sentiment("worst server ever. too many hackers and cheaters. refund please.") # 혹평 (FPS 유저 느낌)
predict_sentiment("it is okay but optimization is bad.") # 애매한 평가